# Teacher Pipeline — Test Notebook

Two-stage pipeline:
1. **Context Analysis (OpenAI `gpt-4o-mini`)**: Political saliency extraction + structured video analysis (`context.parquet`).
2. **Pairwise Scoring (TypeSafe `Jev` via OpenRouter)**: Fast, calibrated decision scoring on Economic and GAL-TAN similarity in `[-1.0, 1.0]`.

In [ ]:
import sys
sys.path.insert(0, '../src')

In [ ]:
import itertools
import random
from dotenv import load_dotenv
load_dotenv('../.env', override=True)

import polars as pl
from config import OUT_DIR, SLANTS_TRAIN
from teacher.context import analyze_video, VideoAnalysis
from teacher.pair_similarity import score_pair

## 1. Inspect Pre-computed `context.parquet` (Batch Outputs)
If you have run `unpack_batch.py`, you can inspect and sample directly from the unpacked dataset:

In [ ]:
context_parquet = OUT_DIR / "context.parquet"

if context_parquet.exists():
    df_context = pl.read_parquet(context_parquet)
    print(f"Total videos in context.parquet: {len(df_context)}")
    df_political = df_context.filter(pl.col("political_saliency") > 0)
    print(f"Political videos: {len(df_political)} ({len(df_political)/len(df_context):.1%})")
    display(df_political.select(["id", "channel", "slant", "political_saliency", "title"]).head(5))
else:
    print("context.parquet not found yet. Run `python -m teacher.unpack_batch` first.")

## 2. Test Live Stage 1 Sample (`analyze_video`)
Test live analysis on a few rows from `slants_train.csv`:

In [ ]:
N_SAMPLES = 4

sample_videos = (
    pl.read_csv(SLANTS_TRAIN, ignore_errors=True)
    .select(["id", "title", "channel", "description", "tags", "slant"])
    .filter(pl.col("id").str.contains(r"^[A-Za-z0-9_-]{11}$"))
    .unique(subset=["id"], keep="first")
    .sample(N_SAMPLES, shuffle=True)
    .to_dicts()
)

print(f"Loaded {len(sample_videos)} sample videos:")
for v in sample_videos:
    print(f"  · [{v['id']}] [{v['channel']}] slant={v.get('slant')} | {v['title'][:55]}")

In [ ]:
# Run single-video analysis live
sample_analyses = []
for v in sample_videos:
    a = analyze_video(v)
    sample_analyses.append(a)
    print(f"  saliency={a['political_saliency']:.2f} | {v['title'][:60]}")

if sample_analyses:
    import json
    print("\nSample Structured Analysis:")
    print(json.dumps(json.loads(sample_analyses[0]["analysis_json"]), indent=2))

## 3. Test Stage 2 Pairwise Scoring (`Jev` via OpenRouter)
Sample political videos and score pairs using Jev's 5-level rubric schema:

In [ ]:
# Load political video pairs
if context_parquet.exists():
    records = df_context.filter(pl.col("political_saliency") > 0).sample(4, shuffle=True).to_dicts()
else:
    records = [a for a in sample_analyses if a["political_saliency"] > 0]

political_sample = [
    (v, VideoAnalysis.model_validate_json(v["analysis_json"]))
    for v in records
]

# Create pairwise combinations
test_pairs = list(itertools.combinations(political_sample, 2))[:3]
print(f"Testing Jev scoring on {len(test_pairs)} pairs...\n")

results = []
for pair in test_pairs:
    res = score_pair(pair)
    results.append(res)
    print(f"Pair: '{res['title_a'][:30]}' × '{res['title_b'][:30]}'")
    print(f"  -> economic_score: {res['economic_score']:+.4f} | galtan_score: {res['galtan_score']:+.4f}")
    print(f"  -> slant_a: {res['slant_a']} | slant_b: {res['slant_b']}\n")

pl.DataFrame(results)

## 4. Cost Estimates
- **Stage 1 (OpenAI `gpt-4o-mini` Batch API)**: 50% discount ($0.075 / 1M prompt, $0.300 / 1M completion) -> ~$2.80 per 50k videos.
- **Stage 2 (TypeSafe `Jev` via OpenRouter)**: ~$0.000019 per pair decision -> ~$0.02 per 1,000 pairs.